# 02 · Fine-tune Gemma 3 1B with LoRA

**Goal:** teach Gemma 3 1B (instruction-tuned) two FairRent tasks:
1. **extract**: read a tenant's message (English, Pidgin, Yoruba) and return fee JSON
2. **reply**: turn the fee engine's result into a short, grounded answer

**Runs on:** Colab **T4 GPU** (Runtime → Change runtime type → T4 GPU). Training takes about 15–30 minutes.

**Before you start:**
1. Accept the Gemma licence at https://huggingface.co/google/gemma-3-1b-it (logged in to Hugging Face).
2. Create a Hugging Face access token (Settings → Access Tokens, "Read").
3. In Colab, open the 🔑 **Secrets** panel, add `HF_TOKEN` with your token, and allow notebook access.

In [ ]:
# --- Setup: get the project code (works in Colab and locally) ---
REPO_URL = "https://github.com/YOUR-GITHUB-USERNAME/fairrent-akure.git"  # <-- change to your repo

import os, sys
if os.path.exists("../src/fairrent"):          # running locally from notebooks/
    os.chdir("..")
elif not os.path.exists("src/fairrent"):       # running in Colab
    folder = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(folder):
        os.system(f"git clone -q {REPO_URL}")
    os.chdir(folder)
sys.path.insert(0, "src")
print("Working in:", os.getcwd())

In [ ]:
!pip install -q "transformers>=4.50" "peft>=0.13" accelerate datasets

In [ ]:
import json, time
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # reduces GPU memory fragmentation
import torch
from huggingface_hub import login

try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception as e:
    print("No Colab secret found; using an existing login if there is one.", e)

assert torch.cuda.is_available(), "Switch the runtime to a GPU (T4)."
print(torch.cuda.get_device_name(0))

## 1. Load the base model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from fairrent.llm import DEFAULT_MODEL_ID, pick_dtype

MODEL_ID = DEFAULT_MODEL_ID          # google/gemma-3-1b-it
dtype = pick_dtype()                  # float32 on T4, bfloat16 on A100/L4
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype,
                                             attn_implementation="eager", device_map="auto")
print(f"Loaded {MODEL_ID} in {dtype}; parameters: {model.num_parameters()/1e9:.2f}B")

## 2. Prepare the training data

Only the assistant's answer is trained on: prompt tokens are masked with `-100`, so the model learns to answer, not to repeat the instructions.

In [ ]:
from datasets import Dataset

MAX_LEN = 1024

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]

def tokenize(example):
    msgs = example["messages"]
    prompt = tokenizer.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    answer = msgs[-1]["content"] + "<end_of_turn>\n"
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
    input_ids = (prompt_ids + answer_ids)[:MAX_LEN]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_LEN]
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}

train_raw, test_raw = load_jsonl("data/sft_train.jsonl"), load_jsonl("data/sft_test.jsonl")
train_ds = Dataset.from_list(train_raw).map(tokenize, remove_columns=["id", "task", "messages"])
eval_ds = Dataset.from_list(test_raw).map(tokenize, remove_columns=["id", "task", "messages"])
lengths = [len(x) for x in train_ds["input_ids"]] + [len(x) for x in eval_ds["input_ids"]]
print(f"train examples: {len(train_ds)}  test examples: {len(eval_ds)}  longest: {max(lengths)} tokens")
assert max(lengths) < MAX_LEN, "An example is too long and would be cut; raise MAX_LEN"

## 3. Add LoRA adapters

LoRA trains small low-rank matrices next to the frozen model weights. With rank 16 only about 1% of the weights are trained,
which suits a small dataset, limits overfitting, and keeps GPU time (and energy use) low.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

model.config.use_cache = False          # required with gradient checkpointing
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Train

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

ADAPTER_DIR = "fairrent-gemma3-1b-lora"
args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=1,       # fits a T4 in float32 (Gemma has a 262k-word vocabulary)
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,       # effective batch size 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="no",
    bf16=(dtype == torch.bfloat16),
    report_to="none",
    label_names=["labels"],
    seed=42,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                  data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))
start = time.time()
trainer.train()
minutes = (time.time() - start) / 60
print(f"Training took {minutes:.1f} minutes")

In [ ]:
import matplotlib.pyplot as plt
os.makedirs("results", exist_ok=True)
log = trainer.state.log_history
train_pts = [(l["step"], l["loss"]) for l in log if "loss" in l]
eval_pts = [(l["step"], l["eval_loss"]) for l in log if "eval_loss" in l]
plt.plot(*zip(*train_pts), label="train loss")
plt.plot(*zip(*eval_pts), "o-", label="test loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("LoRA fine-tuning")
plt.savefig("results/training_loss.png", dpi=120, bbox_inches="tight"); plt.show()

json.dump({"model_id": MODEL_ID, "dtype": str(dtype), "gpu": torch.cuda.get_device_name(0),
           "train_minutes": round(minutes, 1), "lora": {"r": 16, "alpha": 32, "dropout": 0.05},
           "epochs": 3, "learning_rate": 2e-4, "effective_batch": 8,
           "train_examples": len(train_ds), "log_history": log},
          open("results/training_log.json", "w"), indent=2)

## 5. Save the adapter

The adapter is small (tens of MB). Save it to Google Drive so notebook 03 can load it after a restart,
and optionally publish it on Hugging Face so others can reproduce your results.

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/fairrent
    !cp -r {ADAPTER_DIR} /content/drive/MyDrive/fairrent/
    print("Saved to Google Drive: MyDrive/fairrent/" + ADAPTER_DIR)

# Optional: publish the adapter (uncomment and set your username)
# model.push_to_hub("YOUR-HF-USERNAME/fairrent-gemma3-1b-lora")

## 6. Quick check

In [ ]:
from fairrent.llm import generate
from fairrent.prompts import extract_messages

model.eval()
model.config.use_cache = True
msg = "Abeg check am. Selfcon for southgate. Rent na 300k, agency 60k, development fee 40k."
print(generate(model, tokenizer, [extract_messages(msg)], max_new_tokens=200)[0])